### Ingestion pipeline


In [5]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader


### Read all the PDFs inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDFs in a directory"""

    all_documents = []

    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")

        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            # Add all pages/documents
            all_documents.extend(documents)

        except Exception as e:
            print(f"Error processing {pdf_file.name}: {e}")

    return all_documents

In [7]:
documents = process_all_pdfs("../data/pdf")

print(f"\nTotal documents/pages: {len(documents)}")

print(documents[0].metadata)

Found 2 PDF files to process

Processing: ASIM ILYAS CV ML.pdf

Processing: Hybrid Multimodal Fusion Framework for Cardiovascular Disease Detection Final.pdf

Total documents/pages: 105
{'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-08-08T17:36:03+00:00', 'author': '', 'keywords': '', 'moddate': '2026-08-08T17:36:03+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\ASIM ILYAS CV ML.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'ASIM ILYAS CV ML.pdf', 'file_type': 'pdf'}


In [8]:
documents

[Document(metadata={'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-08-08T17:36:03+00:00', 'author': '', 'keywords': '', 'moddate': '2026-08-08T17:36:03+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\ASIM ILYAS CV ML.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'ASIM ILYAS CV ML.pdf', 'file_type': 'pdf'}, page_content='Asim Ilyas\n+92 320 1587154|asimalyas4440@gmail.com |linkedin.com/in/asim-ilyas |github.com/asimalyas |portfolio\nProfessional Summary\nMachine Learning Engineer with a 3.90/4.00 CGPA (97.5%) in Software Engineering from COMSATS University\nIslamabad and one year of hands-on experience engineering and deploying production AI solutions. Delivered 20+\nAI/ML and software projects spanning ensemble learning, explainable AI (XAI), and end-to-end ML pipelines. Ranked\n1st 

### text spliting gget into shunks


In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


def chunk_documents(documents, chunk_size=1000, chunk_overlap=200):
    """
    Split documents into smaller chunks while preserving metadata.

    Args:
        documents: List of LangChain Document objects
        chunk_size: Maximum size of each chunk
        chunk_overlap: Overlap between chunks

    Returns:
        List of chunked Document objects
    """

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", " ", ""]
    )

    chunks = text_splitter.split_documents(documents)

    return chunks 

In [19]:
chunks =chunk_documents(documents,500,100)
len(documents)
len(chunks)
chunks[200]


Document(metadata={'producer': 'Microsoft® Word 2013', 'creator': 'Microsoft® Word 2013', 'creationdate': '2026-08-19T00:32:16+05:00', 'title': 'Hybrid Multimodal Fusion Framework for Cardiovascular Disease Detection', 'author': 'Muhammad Haris', 'moddate': '2026-08-19T00:32:16+05:00', 'source': '..\\data\\pdf\\Hybrid Multimodal Fusion Framework for Cardiovascular Disease Detection Final.pdf', 'total_pages': 103, 'page': 29, 'page_label': '30', 'source_file': 'Hybrid Multimodal Fusion Framework for Cardiovascular Disease Detection Final.pdf', 'file_type': 'pdf'}, page_content='5. User receives logout success notification.  \n6. System redirects user to Login page. \nAlternative Flow \nAF-01 Cancel Logout:  \n• User cancels confirmation dialogue → session continues normally.  \nAF-02 Auto Logout:  \n• Session expires automatically after inactivity timeout → system redirects to \nLogin. \nExceptions E-01 Session expired before manual logout → redirect occurs without \nconfirmation. \nBus

### Embeddings

In [20]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [26]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5149.88it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\WALEED TRADERS\AppData\Local\Temp\ipykernel_2460\2964522620.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### VECTOR SEARCH

In [36]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 547


In [37]:
##convert teh chunks into embeddings and add to the vector store
texts=[doc.page_content for doc in chunks]
texts
## genrate embediings
embedding=embedding_manager.generate_embeddings(texts)
## store in VECtor Db
vectorstore.add_documents(chunks,embedding)

Generating embeddings for 547 texts...


Batches: 100%|██████████| 18/18 [00:26<00:00,  1.48s/it]


Generated embeddings with shape: (547, 384)
Adding 547 documents to vector store...
Successfully added 547 documents to vector store
Total documents in collection: 1094


### retrivial  or ONLINE   QUrey  part of RAG

In [122]:
from typing import List, Dict, Any


class RAGRetriever:
    """Handles query-based retrieval from the vector store."""

    def __init__(
        self,
        vector_store: VectorStore,
        embedding_manager: EmbeddingManager
    ):
        """
        Initialize the retriever.

        Args:
            vector_store: Vector store containing document embeddings.
            embedding_manager: Manager for generating query embeddings.
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(
        self,
        query: str,
        top_k: int = 5,
        score_threshold: float = -100
    ) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query.

        Args:
            query: The search query.
            top_k: Number of top results to return.
            score_threshold: Minimum similarity score.

        Returns:
            List of dictionaries containing retrieved documents and metadata.
        """

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings(
            [query]
        )[0]

        try:
            # Search in vector store
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            retrieved_docs = []

            if results["documents"] and results["documents"][0]:

                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]

                for i, (
                    doc_id,
                    document,
                    metadata,
                    distance
                ) in enumerate(
                    zip(
                        ids,
                        documents,
                        metadatas,
                        distances
                    )
                ):

                    # Convert cosine distance to similarity score
                    similarity_score = 1 - distance

                    # Apply score threshold
                    if similarity_score >= score_threshold:

                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1
                        })

                print(
                    f"Retrieved {len(retrieved_docs)} "
                    f"documents (after filtering)"
                )

            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []


rag_retriever = RAGRetriever(
    vectorstore,
    embedding_manager
)

In [123]:
results = rag_retriever.retrieve(
    "Asim cgpa",
   
)

print(results)

Retrieving documents for query: 'Asim cgpa'
Top K: 5, Score threshold: -100
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 53.91it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)
[{'id': 'doc_aaf3e0d9_0', 'content': 'Asim Ilyas\n+92 320 1587154|asimalyas4440@gmail.com |linkedin.com/in/asim-ilyas |github.com/asimalyas |portfolio\nProfessional Summary\nMachine Learning Engineer with a 3.90/4.00 CGPA (97.5%) in Software Engineering from COMSATS University\nIslamabad and one year of hands-on experience engineering and deploying production AI solutions. Delivered 20+\nAI/ML and software projects spanning ensemble learning, explainable AI (XAI), and end-to-end ML pipelines. Ranked', 'metadata': {'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'moddate': '2026-08-08T17:36:03+00:00', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-08-08T17:36:03+00:00', 'source_file': 'ASIM ILYAS CV ML.pdf', 'page_label': '1', 'keywords': '', 'trapped': '/False', 'page': 0, 'doc_index': 0, 'subject': '', 'content_length': 461, 'file_type

### integgartion RAg with LLM RAG

In [124]:
import os
from dotenv import load_dotenv
load_dotenv()

print(os.getenv("GROQ_API_KEY"))

gsk_REDACTED


In [125]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()

True

In [126]:
from langchain_groq import ChatGroq
import os

llm = ChatGroq(
    api_key=os.getenv("GROQ_API_KEY"),
    model="openai/gpt-oss-120b",
    temperature=0.1,
    max_tokens=512
)

In [129]:
def ragsimple(query, rag_retriever, llm, top_k=3):

    # Retrieve relevant documents
    results = rag_retriever.retrieve(
        query,
        top_k=top_k,
        score_threshold=-100
    )

    # Combine retrieved documents
    content = "\n\n".join(
        result["content"]
        for result in results
    )

    # Create prompt
    prompt = f"""
Answer the question using only the provided context.

Context:
{content}

Question:
{query}

Answer:
"""

    # Send prompt to LLM
    response = llm.invoke(prompt)

    return response.content

In [130]:
ans = ragsimple(
    "what is asim FYP ",
    rag_retriever,
    llm,
    top_k=3
)

print(ans)

Retrieving documents for query: 'what is asim FYP '
Top K: 3, Score threshold: -100
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 56.92it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


**ASIM FYP** is a final‑year‑project implementation that builds an end‑to‑end ECG analysis system.  
The project’s pipeline works as follows:

1. **Frontend upload** – the user sends an ECG image file to the backend.  
2. **Pre‑validation** – the image’s structural similarity (SSIM) is computed; an SSIM ≥ 0.70 lets the image continue to full analysis, while a non‑ECG or corrupted file triggers a controlled termination with a clear error message.  
3. **Processing pipeline** – a valid ECG image is passed to the analysis stages (P‑wave, QRS, T‑wave annotation, classifier confidence scores, and clinical prediction explanations).  
4. **Streaming UI progress** – the backend streams progress back to the UI via a POST /predict request, allowing the user to see the analysis in real time.  

The project also includes secure, role‑based modules (Admin, Doctor, Technician, Patient) for ECG upload, approval workflow, report generation, and patient access. A full‑pipeline visualization is availabl

### RAG advanced

In [131]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output


In [133]:
# Example usage:
result = rag_advanced("what is asim FYP?", rag_retriever, llm, top_k=3, min_score=-100, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'what is asim FYP?'
Top K: 3, Score threshold: -100
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 43.11it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: **ASIM FYP** is a university “Final Year Project” (FYP) named **ASIM** that implements an automated pipeline for ECG‑image analysis. In the project the frontend uploads an image, a preliminary validation step computes the Structural Similarity Index (SSIM) and only images with an SSIM ≥ 0.70 are passed on for full ECG processing; non‑ECG or corrupted files trigger a controlled termination with a clear error message. The system then streams progress back to the UI while the backend runs the prediction.
Sources: [{'source': 'Hybrid Multimodal Fusion Framework for Cardiovascular Disease Detection Final.pdf', 'page': 89, 'score': -0.356239914894104, 'preview': 'Frontend sends file data and \nreceives the preliminary validation \nresult. \nPass \n7 Validation to \nprocessing \npipeline \nUse an ECG image with \nSSIM score greater than or \nequal to 0.70. \nAccepted image proceeds to full \nanalysis. \nPass \n8 Invalid input to \ncontrolled \ntermination \nUse a non-ECG ima...'}, {'s

In [ ]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("Who is asim", top_k=3, min_score=-10, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'Who is asim'
Top K: 3, Score threshold: -100
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 58.85it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
Asim Ilyas
+92 320 1587154|asimalyas4440@gmail.com |linkedin.com/in/asim-ilyas |github.com/asimalyas |portfolio
Professional Summary
Machine Learning Engineer with a 3.90/4.00 CGPA (97.5%) in Software Engineering from COMSATS University
Islamabad and 

one year of hands-on experience engineering and deploying production AI solutions. Delivered 20+
AI/ML and software projects spanning ensemble learning, explainable AI (XAI), and end-to-end ML pipelines. Ranked

Asim Ilyas
+92 320 1587154|asimalyas4440@gmail.com |linkedin.com/in/asim-ilyas |github.com/asimalyas |portfolio
Professional Summary
Machine Learning Engineer with a 3.90/4.00 CGPA (97.5%) in Software Engineering from COMSATS University
Islamabad and one year of hands-on experience engineering and deploying production AI solutions. Delivered 20+
AI/ML and software projects spanning ensemble learning, explainable AI (XAI), and end-to-end ML pipelines. Ranked

have been a source of strength throughout this journey. 
 
 
Ferdous Gulzar                                                     Muhammad Asim Ilyas

Question: Who is asim

Answer:

Final Answer: Asim Ilyas is a Machine Learning Engineer who graduated with a 3.90 / 4.00 CGPA (97.5%) in Software Engineering from COMSATS Unive